# Generate Met Forcing: Lower Sycamore Wash (NLDAS-2 point stations)

This notebook builds the **weather files** tRIBS needs for the Lower Sycamore Wash run: temperature, humidity, wind, solar radiation and air pressure, hour by hour, for water year 2017 (Oct 2016 to Sep 2017). **Rain is not made here.** Rain comes from the radar ASCII grids you already built.

**What it does, in order**
1. Finds your `lab` and `init_data` folders.
2. Shows how the basin sits on the NLDAS-2 grid and picks a few cells to act as weather stations.
3. Downloads hourly NLDAS-2 data for each station from NASA (in 30-day pieces, saved so a re-run never repeats work).
4. Checks and cleans the data, converts it to tRIBS units and Arizona time, and writes the station files.
5. Reads the files back and runs sanity checks.

**You get:** `../init_data/met/Master_Met.sdf` plus one `met_NLDAS_<n>_2016-2017.mdf` per station.

**Before you run**
- You need your Earthdata login with the **NASA GESDISC DATA ARCHIVE** app authorized (you already fixed this). If a login prompt appears, type your username and password into the prompt. **Never type them into a code cell** and never commit them.
- The first run downloads 3 stations x 13 pieces x 7 variables, so it takes several minutes. Pieces already saved in `~/nldas_cache` are skipped, so if the connection drops, just run the cell again.
- Run the cells top to bottom.
- A `TqdmWarning: IProgress not found` message when the packages load is harmless.

**Words you will see**
- **UTC**: world standard time. NLDAS is stamped in UTC. Arizona is UTC-7 all year.
- **NLDAS cell**: NLDAS-2 is a grid of 0.125-degree squares (about 12 x 14 km here). Every point inside a cell gets that cell's weather.
- **Thiessen polygons**: tRIBS gives each mesh node the data of its *nearest* station. The set of nodes closest to one station is that station's polygon.
- **Lapse rate**: how fast air cools as you go up, about 6.5 degrees C per km. NLDAS adjusts each cell's temperature to the cell's *average* elevation.
- **Cache**: a saved copy of a download.

## 1. Setup: packages and folders

In [ ]:
import os, io, time, contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pyproj
import rasterio
from rasterio.features import geometry_mask
from rasterio.enums import Resampling
from shapely.geometry import box

from pytRIBS.classes import Met

# Find the `lab` folder (the one that sits next to `init_data`) and work from it.
# That way every path below is written exactly as in Make_Sycamore_Model.ipynb,
# and the station file records paths that tRIBS can follow when it runs from `lab`.
here = Path.cwd().resolve()
LAB = None
for p in [here, *here.parents]:
    if p.name != 'init_data' and (p.parent / 'init_data').is_dir():
        LAB = p
        break
    if (p / 'init_data').is_dir() and (p / 'lab').is_dir():
        LAB = p / 'lab'
        break
if LAB is None:
    raise FileNotFoundError(
        f'Could not find an init_data folder next to a lab folder, starting from {here}. '
        'Open this notebook from workspaces/Sycamore/lab (or a folder inside it).')
os.chdir(LAB)
print('Working from:', Path.cwd())
print('init_data is: ', Path('../init_data').resolve())

## 2. Settings

Everything you might want to change is in this one cell.

- The run window is written in **Arizona clock time**. Radar rain files start at 10/01/2016 00:00, so do not start earlier unless you also pull more radar hours.
- `STATIONS` lists the NLDAS cells that become stations, **lowest to highest**. The first becomes `NLDAS_1`, the second `NLDAS_2`, and so on. Fewer stations run faster; more stations follow the mountains better. Josh suggested a few, so we start with three.
- The cell elevations come from `Met.get_nldas_elevation`, printed on 2026-10-12 (mean, minimum, maximum in metres).
- `LABEL_SHIFT_HOURS` stays 0 until we settle what an NLDAS timestamp means (section 8 tests this).

In [ ]:
NAME = 'Sycamore'
EPSG = 26912                          # UTM zone 12N

# Run window in ARIZONA clock time (MST = UTC-7 all year, no daylight saving)
LOCAL_START = '2016-10-01 00:00'      # first hour of water year 2017
LOCAL_END   = '2017-09-30 23:00'      # last hour
LOCAL_UTC_OFFSET = -7

# Extra hours to move every timestamp. Keep 0 until the hour-label question is settled.
LABEL_SHIFT_HOURS = -1

# NLDAS cells that touch the basin:
#   name: (cell-center latitude, longitude, mean, min, max elevation of the cell in m)
NLDAS_CELLS = {
    'south-west':  (33.6875, -111.5625,  657,  486,  975),
    'south-east':  (33.6875, -111.4375,  984,  609, 1808),
    'middle-west': (33.8125, -111.5625,  892,  505, 1422),
    'middle-east': (33.8125, -111.4375, 1247,  670, 1889),
    'north-west':  (33.9375, -111.5625, 1144,  514, 1954),
    'north-east':  (33.9375, -111.4375, 1435,  997, 2072),
}
STATIONS = ['south-west', 'middle-east', 'north-east']      # low -> high

# Where files go
MET_SDF    = '../init_data/met/Master_Met.sdf'              # used by the model
PRECIP_SDF = '~/nldas_unused_precip/Precip_NLDAS_unused.sdf'  # NOT used (radar is the rain source)
CACHE_DIR  = '~/nldas_cache'                                # saved downloads, outside the repo
CHUNK_DAYS = 30                                             # days per download request

# ---- quick checks on the settings ----
bad = [s for s in STATIONS if s not in NLDAS_CELLS]
if bad:
    raise ValueError(f'STATIONS has names that are not in NLDAS_CELLS: {bad}')
n_hours = len(pd.date_range(LOCAL_START, LOCAL_END, freq='h'))
print(f'Run window: {LOCAL_START} to {LOCAL_END} Arizona time = {n_hours} hourly rows per station')
print('Stations (low to high):', STATIONS)

## 3. Where the basin sits on the NLDAS grid, and where the stations go

This cell reads the basin outline and the DEM, sorts every basin pixel into its NLDAS cell, and prints two tables.

- **Table 1** shows how much of the basin is in each cell, and how the cell's average elevation compares with the real ground in the basin part of that cell. The last column says how much cooler that ground probably is than NLDAS's temperature, using 6.5 degrees C per km (a negative number means warmer).
- **Table 2** shows what each chosen station would serve once tRIBS hands each node to its nearest station.

The station is placed at the basin pixel closest to the middle of the basin part of its cell. Every spot inside the cell gets the same NLDAS data, so the position matters only for the nearest-station rule.

In [ ]:
BOUNDARY = f'data/preprocessing/{NAME}_boundary.shp'
DEM      = f'../init_data/{NAME}_DEM.tif'
for f in (BOUNDARY, DEM):
    if not os.path.exists(f):
        raise FileNotFoundError(f'Cannot find {f} (looking from {Path.cwd()}). '
                                'The boundary comes from Make_Sycamore_Model.ipynb; the DEM is not stored in git.')

# Basin outline (keep only the main piece, as in the mesh notebook)
gdf = gpd.read_file(BOUNDARY).to_crs(EPSG)
geom = gdf.geometry.union_all() if hasattr(gdf.geometry, 'union_all') else gdf.geometry.unary_union
if geom.geom_type == 'MultiPolygon':
    geom = max(geom.geoms, key=lambda g: g.area)

# Read the DEM at 1/10 resolution (10 m -> 100 m). That is plenty for averages and fast.
SUB = 10
with rasterio.open(DEM) as src:
    h, w = src.height // SUB, src.width // SUB
    z = src.read(1, out_shape=(h, w), resampling=Resampling.nearest, masked=True)
    tf = src.transform * src.transform.scale(src.width / w, src.height / h)

inside = geometry_mask([geom], out_shape=(h, w), transform=tf, invert=True) & ~np.ma.getmaskarray(z)
rr, cc = np.nonzero(inside)
px = pd.DataFrame({'x': tf.c + tf.a * (cc + 0.5) + tf.b * (rr + 0.5),
                   'y': tf.f + tf.d * (cc + 0.5) + tf.e * (rr + 0.5),
                   'z': np.asarray(z.data, dtype=float)[rr, cc]})
to_ll = pyproj.Transformer.from_crs(EPSG, 4326, always_xy=True)
px['lon'], px['lat'] = to_ll.transform(px['x'].to_numpy(), px['y'].to_numpy())
PX_KM2 = abs(tf.a * tf.e) / 1e6
total_km2 = len(px) * PX_KM2
print(f'Basin: {geom.area / 1e6:.1f} km2 from the outline; {total_km2:.1f} km2 counted from {len(px):,} DEM pixels')

# Sort every basin pixel into its NLDAS cell (a cell reaches 0.0625 degrees from its center)
HALF = 0.0625
px['cell'] = ''
for name, (clat, clon, *_rest) in NLDAS_CELLS.items():
    m = (px['lat'] >= clat - HALF) & (px['lat'] < clat + HALF) & \
        (px['lon'] >= clon - HALF) & (px['lon'] < clon + HALF)
    px.loc[m, 'cell'] = name
other_pct = 100 * (px['cell'] == '').mean()

# Table 1: every candidate cell
LAPSE = 6.5 / 1000                                   # degrees C per metre
rows = []
for name, (clat, clon, emean, emin, emax) in NLDAS_CELLS.items():
    sub = px[px['cell'] == name]
    ground = sub['z'].mean() if len(sub) else np.nan
    rows.append({'cell': name, 'basin km2': len(sub) * PX_KM2, 'basin %': 100 * len(sub) / len(px),
                 'NLDAS cell mean m': emean, 'basin ground mean m': ground,
                 'ground minus cell m': ground - emean, 'cooler by C (neg = warmer)': (ground - emean) * LAPSE})
table1 = pd.DataFrame(rows).set_index('cell')
print('\nTable 1: NLDAS cells and the basin')
display(table1.round(1))
print(f'Basin area in cells that are NOT listed in NLDAS_CELLS: {other_pct:.1f}%  (should be small)')

# Pick the stations
stations = []
for i, name in enumerate(STATIONS, start=1):
    sub = px[px['cell'] == name]
    if sub.empty:
        raise ValueError(f"Cell '{name}' has no basin area. Remove it from STATIONS.")
    cx, cy = sub['x'].mean(), sub['y'].mean()
    k = ((sub['x'] - cx) ** 2 + (sub['y'] - cy) ** 2).idxmin()
    r = sub.loc[k]
    stations.append({'n': i, 'label': name, 'x': round(float(r['x']), 1), 'y': round(float(r['y']), 1),
                     'lat': float(r['lat']), 'lon': float(r['lon']),
                     'elev': int(NLDAS_CELLS[name][2]), 'dem_here': float(r['z'])})

# Table 2: which part of the basin each station serves (nearest station wins)
sx = np.array([s['x'] for s in stations]); sy = np.array([s['y'] for s in stations])
d2 = (px['x'].to_numpy()[:, None] - sx) ** 2 + (px['y'].to_numpy()[:, None] - sy) ** 2
px['station'] = d2.argmin(axis=1)
rows = []
for i, s in enumerate(stations):
    sub = px[px['station'] == i]
    ground = sub['z'].mean()
    rows.append({'station': f"NLDAS_{s['n']}", 'cell': s['label'], 'easting': s['x'], 'northing': s['y'],
                 'station elev m': s['elev'], 'basin km2 served': len(sub) * PX_KM2,
                 'basin % served': 100 * len(sub) / len(px), 'ground mean m served': ground,
                 'ground minus station m': ground - s['elev'], 'cooler by C (neg = warmer)': (ground - s['elev']) * LAPSE})
table2 = pd.DataFrame(rows).set_index('station')
print('\nTable 2: what each station would serve')
display(table2.round(1))

# Map
fig, ax = plt.subplots(figsize=(6.5, 8))
sc = ax.scatter(px['x'], px['y'], c=px['station'], s=2, cmap='Set2', vmin=0, vmax=7)
gpd.GeoSeries([geom], crs=EPSG).boundary.plot(ax=ax, color='k', linewidth=0.8)
for name, (clat, clon, *_rest) in NLDAS_CELLS.items():
    cell = gpd.GeoSeries([box(clon - HALF, clat - HALF, clon + HALF, clat + HALF)], crs=4326).to_crs(EPSG)
    cell.boundary.plot(ax=ax, color='gray', linewidth=0.6, linestyle='--')
for s in stations:
    ax.plot(s['x'], s['y'], marker='*', color='crimson', markersize=16, markeredgecolor='k')
    ax.annotate(f"NLDAS_{s['n']}\n{s['elev']} m", (s['x'], s['y']), xytext=(8, 6), textcoords='offset points', fontsize=8)
ax.set_aspect('equal'); ax.set_title('Basin colored by nearest station (dashed = NLDAS cells)')
ax.set_xlabel('Easting (m)'); ax.set_ylabel('Northing (m)')
plt.show()

## 4. Download NLDAS-2 for each station

The download function that comes with pytRIBS (`get_nldas_point`) fetches one point for one date range. This cell wraps it so that:
- the request is split into **30-day pieces** (`CHUNK_DAYS`), because a failed request then costs minutes, not the whole year;
- each finished piece is **saved** in `CACHE_DIR`, and a re-run skips saved pieces;
- a piece is **retried** up to 3 times, and a piece that comes back missing any of the 7 variables counts as failed (pytRIBS prints an error for a missing variable but carries on, which would otherwise leave holes without warning);
- the dates asked for reach **one day before and one day after** your run window, because NLDAS is in UTC and the shift to Arizona time pulls in hours from the neighbouring days.

If a piece still fails after 3 tries, the cell stops and says which one. Run it again and it resumes where it stopped.

In [ ]:
NEEDED = ['prcp', 'temp', 'humidity', 'psurf', 'wind_u', 'wind_v', 'rsds']

first_utc_day = pd.Timestamp(LOCAL_START).normalize() - pd.Timedelta(days=1)
last_utc_day  = pd.Timestamp(LOCAL_END).normalize()   + pd.Timedelta(days=1)

def make_chunks(first_day, last_day, n_days):
    """Split first_day..last_day into (start, end) date strings, n_days at a time."""
    out, d = [], first_day
    while d <= last_day:
        e = min(d + pd.Timedelta(days=n_days - 1), last_day)
        out.append((d.strftime('%Y-%m-%d'), e.strftime('%Y-%m-%d')))
        d = e + pd.Timedelta(days=1)
    return out

def cache_file(st, start, end):
    # keyed by the station's latitude/longitude, so moving a station triggers a fresh download
    return Path(CACHE_DIR).expanduser() / f"nldas_{st['lat']:.4f}_{st['lon']:.4f}_{start}_{end}.csv"

def fetch_chunk(getter, st, start, end, tries=3, wait=10):
    """Return hourly NLDAS data (UTC) for one station and one date range, using the cache if it can."""
    path = cache_file(st, start, end)
    if path.exists():
        return pd.read_csv(path, index_col=0, parse_dates=True)
    hours = pd.date_range(f'{start} 00:00', f'{end} 23:00', freq='h')
    problem = ''
    for attempt in range(1, tries + 1):
        log = io.StringIO()
        try:
            with contextlib.redirect_stdout(log):          # keep pytRIBS's long banner out of the output
                df = getter([(st['x'], st['y'])], start, end)
            notes = [ln.strip() for ln in log.getvalue().splitlines() if 'Error' in ln or 'Warning' in ln]
            df = df.copy()
            df.index = pd.to_datetime(df.index)
            if df.index.tz is not None:
                df.index = df.index.tz_convert('UTC').tz_localize(None)
            df = df[~df.index.duplicated()]
            absent = [c for c in NEEDED if c not in df.columns]
            if absent:
                problem = f'missing variables {absent}. {" ".join(notes[:2])}'
            elif len(df) == 0:
                problem = 'no rows came back'
            else:
                df = df[NEEDED].reindex(hours)             # a missing hour becomes an empty row, fixed later
                df.index.name = 'time'
                path.parent.mkdir(parents=True, exist_ok=True)
                df.to_csv(path)
                return df
        except Exception as e:
            problem = f'{type(e).__name__}: {e}'
        print(f'   attempt {attempt} of {tries} failed for {start} to {end}: {problem}')
        if attempt < tries:
            time.sleep(wait)
    raise RuntimeError(f'Giving up on {st["label"]} {start} to {end} after {tries} tries: {problem}\n'
                       'Run this cell again to resume (finished pieces are saved). If it keeps failing, '
                       'check your Earthdata login and that "NASA GESDISC DATA ARCHIVE" is authorized.')

chunks = make_chunks(first_utc_day, last_utc_day, CHUNK_DAYS)
todo = [(s['label'], c) for s in stations for c in chunks if not cache_file(s, *c).exists()]
print(f'{len(stations)} stations x {len(chunks)} pieces = {len(stations) * len(chunks)} requests; '
      f'{len(todo)} still to download, {len(stations) * len(chunks) - len(todo)} already saved.')

met = Met()
getter = lambda pts, b, e: met.get_nldas_point(pts, b, e, epsg=EPSG)
if todo:
    import earthaccess
    earthaccess.login(persist=True)      # one visible login, so any prompt shows up here

raw = {}
t0 = time.time()
for s in stations:
    parts = []
    for (a, b) in chunks:
        if not cache_file(s, a, b).exists():
            print(f"  {s['label']}: downloading {a} to {b} ...")
        parts.append(fetch_chunk(getter, s, a, b))
    raw[s['label']] = pd.concat(parts).sort_index()
    print(f"{s['label']}: {len(raw[s['label']]):,} hourly rows in hand")
print(f'Done in {(time.time() - t0) / 60:.1f} minutes.')

## 5. Check the raw data and fill small holes

NLDAS should have every hour. This cell:
- lines each station up on a complete hourly UTC clock and counts any hours that are missing;
- fills holes of up to 3 hours by straight-line interpolation (anything longer stops the notebook, so you can look);
- sets missing rain to 0, which is harmless because the model uses the radar rain, not this column;
- checks the **units** of the raw numbers (temperature in kelvin, pressure in pascals, humidity in kg/kg), so a download in the wrong units cannot slip through.

In [ ]:
full_hours = pd.date_range(first_utc_day, last_utc_day + pd.Timedelta(hours=23), freq='h')

rows = []
for s in stations:
    df = raw[s['label']].reindex(full_hours)
    n_missing = int(df.isna().any(axis=1).sum())
    df['prcp'] = df['prcp'].fillna(0.0)
    df = df.interpolate(limit=3, limit_area='inside')
    left = int(df.isna().any(axis=1).sum())
    rows.append({'station': f"NLDAS_{s['n']}", 'cell': s['label'], 'hours expected': len(df),
                 'hours with a gap': n_missing, 'still empty after fill': left})
    if left:
        bad_hours = df.index[df.isna().any(axis=1)]
        raise ValueError(f"{s['label']}: {left} hours are still empty after filling (first: {bad_hours[0]}). "
                         'The gap is longer than 3 hours; look at the cache file or re-download.')
    # unit sanity on the raw numbers
    assert 240 < df['temp'].mean() < 320,        f"{s['label']}: temp does not look like kelvin"
    assert 60000 < df['psurf'].mean() < 110000,  f"{s['label']}: psurf does not look like pascals"
    assert 0 < df['humidity'].max() < 0.05,      f"{s['label']}: humidity does not look like kg/kg"
    assert 0 <= df['rsds'].min() and df['rsds'].max() < 1500, f"{s['label']}: rsds out of range"
    s['data'] = df
display(pd.DataFrame(rows).set_index('station'))
print('Units look right (kelvin, pascals, kg/kg, W/m2). Rain total, for reference only (NLDAS mm):',
      {f"NLDAS_{s['n']}": round(float(s['data']['prcp'].sum())) for s in stations})

## 6. Convert to tRIBS units and write the station files

pytRIBS does the conversion in `convert_and_write_nldas_timeseries`:

| NLDAS | becomes | how |
|---|---|---|
| pressure, Pa | `PA`, hPa (mb) | x 0.01 |
| temperature, K | `TA`, degrees C | minus 273.15 |
| humidity, kg/kg | `RH`, % | via vapour pressure |
| wind u and v at 10 m | `US`, m/s at 2 m | vector length, then a wind-profile factor |
| solar, W/m2 | `IS`, W/m2 | as is |

It also moves the timestamps from UTC to Arizona time (`gmt`), trims to your run window, and writes **two** sets of files: weather (used) and NLDAS rain. The rain files go to a folder **outside the repo** (`PRECIP_SDF`) so they can never be picked up by a radar run.

Each station is written with its **NLDAS cell mean elevation**, because that is the elevation NLDAS used to adjust the temperature and pressure. **No extra lapse-rate correction is applied here**; whether and how to do that is on the question list for Josh.

In [ ]:
met_sdf    = MET_SDF
precip_sdf = os.path.expanduser(PRECIP_SDF)
os.makedirs(os.path.dirname(met_sdf), exist_ok=True)
os.makedirs(os.path.dirname(precip_sdf), exist_ok=True)

station_coords = [[s['lon'], s['x'], s['lat'], s['y'], s['elev']] for s in stations]   # lon, easting, lat, northing, elev
met.convert_and_write_nldas_timeseries(
    [s['data'].copy() for s in stations],          # copies: the converter edits what it is given
    station_coords,
    gmt=LOCAL_UTC_OFFSET + LABEL_SHIFT_HOURS,
    met_path=met_sdf, precip_path=precip_sdf,
    orig_begin=LOCAL_START, orig_end=LOCAL_END)

print('Station numbers in the files:')
for s in stations:
    print(f"  NLDAS_{s['n']}  = {s['label']:12s} cell, easting {s['x']}, northing {s['y']}, elevation {s['elev']} m")
print('\nWeather files in', os.path.dirname(met_sdf), ':')
for f in sorted(os.listdir(os.path.dirname(met_sdf))):
    print('  ', f)

# Warn about old files from an earlier run that the new station file no longer uses
listed = {Path(st['file_path']).name for st in Met.read_sdf(met_sdf)}
stale = [f for f in os.listdir(os.path.dirname(met_sdf)) if f.startswith('met_') and f.endswith('.mdf') and f not in listed]
if stale:
    print('\nNOTE: these .mdf files are not listed in Master_Met.sdf (left over from an earlier run?):', stale)

## 7. Read the files back and check them

This reads the files from disk (not from memory), so it checks what tRIBS will actually see. It reports problems in a list at the end; an empty list is what you want.

Checks: row count and no gaps or repeats in the hourly clock; first and last hour; no empty values; each variable inside a believable range; pressure close to what the station's elevation predicts; and the **temperature drop per km** between stations, which should be near the 6.5 degrees C per km that NLDAS builds in.

In [ ]:
problems = []
sdf_rows = Met.read_sdf(met_sdf)
if len(sdf_rows) != len(stations):
    problems.append(f'sdf lists {len(sdf_rows)} stations, expected {len(stations)}')

back = {}
summary = []
for s, row in zip(stations, sdf_rows):
    path = row['file_path']
    if not os.path.exists(path):
        problems.append(f'{path} listed in the sdf but not found from {Path.cwd()}')
        continue
    d = Met.read_met_station(path).set_index('date')
    back[s['label']] = d
    tag = f"NLDAS_{s['n']}"
    expected_idx = pd.date_range(LOCAL_START, LOCAL_END, freq='h')
    if len(d) != len(expected_idx):                 problems.append(f'{tag}: {len(d)} rows, expected {len(expected_idx)}')
    if not d.index.equals(expected_idx):            problems.append(f'{tag}: hourly clock has gaps, repeats or the wrong start/end')
    if d[['PA', 'RH', 'US', 'TA', 'IS']].isna().any().any(): problems.append(f'{tag}: empty values present')
    if not d['TA'].between(-15, 50).all():          problems.append(f"{tag}: TA outside -15..50 C ({d['TA'].min():.1f}..{d['TA'].max():.1f})")
    if not d['RH'].between(0, 105).all():           problems.append(f"{tag}: RH outside 0..105 % ({d['RH'].min():.1f}..{d['RH'].max():.1f})")
    if not d['US'].between(0, 30).all():            problems.append(f"{tag}: US outside 0..30 m/s")
    if not d['IS'].between(0, 1250).all():          problems.append(f"{tag}: IS outside 0..1250 W/m2")
    pa_std = 1013.25 * (1 - 2.25577e-5 * s['elev']) ** 5.25588      # standard atmosphere at the station elevation
    if abs(d['PA'].mean() - pa_std) > 12:           problems.append(f"{tag}: mean PA {d['PA'].mean():.0f} mb vs {pa_std:.0f} expected at {s['elev']} m")
    summary.append({'station': tag, 'elev m': s['elev'], 'rows': len(d),
                    'first hour': d.index[0].strftime('%Y-%m-%d %H:%M'), 'last hour': d.index[-1].strftime('%Y-%m-%d %H:%M'),
                    'TA mean C': d['TA'].mean(), 'RH mean %': d['RH'].mean(), 'RH>100 hours': int((d['RH'] > 100).sum()),
                    'US mean m/s': d['US'].mean(), 'IS mean W/m2': d['IS'].mean(), 'PA mean mb': d['PA'].mean(), 'PA expected mb': pa_std})
display(pd.DataFrame(summary).set_index('station').round(1))

if len(summary) >= 2:
    e = np.array([r['elev m'] for r in summary]); t = np.array([r['TA mean C'] for r in summary])
    drop = -np.polyfit(e, t, 1)[0] * 1000
    print(f'Temperature drop with elevation between the stations: {drop:.1f} C per km (NLDAS builds in about 6.5)')

# Plots: daily means, plus one summer week of hourly sunshine
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for s in stations:
    d = back.get(s['label'])
    if d is None: continue
    lab = f"NLDAS_{s['n']} ({s['elev']} m)"
    d['TA'].resample('D').mean().plot(ax=axs[0, 0], label=lab)
    d['RH'].resample('D').mean().plot(ax=axs[0, 1], label=lab)
    d['US'].resample('D').mean().plot(ax=axs[1, 0], label=lab)
    d['IS'].loc['2017-06-18':'2017-06-24'].plot(ax=axs[1, 1], label=lab)
for ax, ttl in zip(axs.ravel(), ['Air temperature, daily mean (C)', 'Relative humidity, daily mean (%)',
                                 'Wind at 2 m, daily mean (m/s)', 'Solar radiation, hourly, 18-24 Jun 2017 (W/m2)']):
    ax.set_title(ttl); ax.set_xlabel('')
axs[0, 0].legend(fontsize=8); plt.tight_layout(); plt.show()

print('\nPROBLEMS FOUND:' if problems else '\nNo problems found.')
for p in problems: print('  -', p)

## 8. Which hour does an NLDAS timestamp mean?

Open question: the stamp `15:00` could mean the hour that **starts** at 15:00 or the hour that **ends** at 15:00. If it means *ending*, every weather value is one hour later than tRIBS may assume. For streamflow this is a small effect (it moves the timing of evaporation, not the storm), but it should be settled, together with the same question for the radar files.

This test asks the sun. For each candidate window, it computes the average *clear-sky* sunshine over the window, then checks which window best matches the observed shape of sunshine on the clearest days of the year. Sunrise and sunset are sharp, so an hour of error shows up clearly.

- Best window start near **-60 minutes** (relative to the stamp) means the stamp marks the **end** of the hour.
- Best window start near **0** means the stamp marks the **start** of the hour.

This is evidence, not proof: it assumes the solar column is an hourly average. Take the result to Josh. If the answer turns out to be "end of hour" and tRIBS wants start-of-hour, set `LABEL_SHIFT_HOURS = -1` in section 2 and rerun from there.

In [ ]:
def sun_cosz(times, lat, lon):
    """Cosine of the sun's zenith angle for UTC times (NOAA low-precision formulas)."""
    t = pd.DatetimeIndex(times)
    doy = t.dayofyear.to_numpy()
    hr = t.hour.to_numpy() + t.minute.to_numpy() / 60 + t.second.to_numpy() / 3600
    g = 2 * np.pi / 365 * (doy - 1 + (hr - 12) / 24)
    eot = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                    - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))              # minutes
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    ha = np.deg2rad((hr * 60 + eot + 4 * lon) / 4 - 180)
    la = np.deg2rad(lat)
    return np.sin(la) * np.sin(decl) + np.cos(la) * np.cos(decl) * np.cos(ha)

def haurwitz(cosz):
    """Clear-sky sunshine on a flat surface (W/m2) from cos(zenith)."""
    c = np.clip(cosz, 0, None)
    out = np.zeros_like(c, dtype=float)
    up = c > 0
    out[up] = 1098 * c[up] * np.exp(-0.059 / c[up])
    return out

mid = stations[len(stations) // 2]
labels = mid['data'].index                                  # UTC stamps, as downloaded
obs = mid['data']['rsds'].to_numpy()

minutes = pd.date_range(labels[0] - pd.Timedelta(hours=3), labels[-1] + pd.Timedelta(hours=3), freq='min')
cs_min = haurwitz(sun_cosz(minutes + pd.Timedelta(seconds=30), mid['lat'], mid['lon']))
csum = np.concatenate([[0.0], np.cumsum(cs_min)])
start_idx = np.asarray((labels - minutes[0]) / pd.Timedelta(minutes=1), dtype=int)

def clear_sky(offset_min):
    """Average clear-sky sunshine over the hour that starts offset_min minutes after each stamp."""
    i0 = start_idx + offset_min
    return (csum[i0 + 60] - csum[i0]) / 60.0

day_id, _ = pd.factorize((labels + pd.Timedelta(hours=LOCAL_UTC_OFFSET)).normalize())

def day_fit(cs):
    """Per local day: best scale factor k and the relative misfit of obs vs k * clear-sky."""
    num = np.bincount(day_id, weights=obs * cs); den = np.bincount(day_id, weights=cs * cs)
    k = np.divide(num, den, out=np.zeros_like(num), where=den > 0)
    resid = np.bincount(day_id, weights=(obs - k[day_id] * cs) ** 2)
    ref = np.bincount(day_id, weights=obs ** 2)
    rel = np.sqrt(np.divide(resid, ref, out=np.full_like(resid, np.nan), where=ref > 0))
    return k, rel

k0, _ = day_fit(clear_sky(-45))
clear_days = (k0 > 0.85) & (k0 < 1.25)
if clear_days.sum() < 15:                                   # fall back to the clearest quarter of days
    clear_days = k0 >= np.nanpercentile(k0, 75)

scores = []
for off in range(-90, 31, 15):
    k, rel = day_fit(clear_sky(off))
    scores.append({'window starts (min after stamp)': off, 'median misfit': float(np.nanmedian(rel[clear_days]))})
scores = pd.DataFrame(scores).set_index('window starts (min after stamp)')
best = int(scores['median misfit'].idxmin())
display(scores.round(3))
print(f'Used {int(clear_days.sum())} clear days at {mid["label"]} ({mid["lat"]:.3f} N, {mid["lon"]:.3f} E).')
print(f'Best-fit window starts {best} minutes after the stamp  (-60 = hour ENDING at the stamp, 0 = hour STARTING at the stamp).')
print('Closer to the END-of-hour convention.' if abs(best + 60) < abs(best) else 'Closer to the START-of-hour convention.')

## 9. Record where the data came from, and what to do next

Paste the block this prints into the "Other weather" section of `init_data/data_sources.md`. The next cell lists what to change in `Make_Sycamore_Model.ipynb`.

In [ ]:
today = pd.Timestamp.now().strftime('%Y-%m-%d')
print(f"""### Other weather: NLDAS-2 point stations (written {today})
- Source: NLDAS-2 hourly primary forcing (NLDAS_FORA0125_H v2.0, 0.125 degree), NASA GES DISC, retrieved through the Giovanni time-series service with pytRIBS Met.get_nldas_point (pytRIBS 1.0.0).
- Variables: air temperature, specific humidity, surface pressure, 10 m wind (u and v), downward shortwave radiation. NLDAS rain was also downloaded but is NOT used (rain comes from the MRMS/IEM radar grids).
- Window: {LOCAL_START} to {LOCAL_END} Arizona time (UTC{LOCAL_UTC_OFFSET:+d}); requested {first_utc_day:%Y-%m-%d} to {last_utc_day:%Y-%m-%d} UTC. Timestamp shift: gmt = {LOCAL_UTC_OFFSET + LABEL_SHIFT_HOURS} (LABEL_SHIFT_HOURS = {LABEL_SHIFT_HOURS}; start-versus-end-of-hour meaning NOT yet confirmed).
- Conversion: pytRIBS convert_and_write_nldas_timeseries (Pa to hPa, K to C, specific humidity to RH, 10 m wind to 2 m, UTC to local). Gaps up to 3 hours filled by linear interpolation.
- Stations (nearest-station rule, no extra lapse-rate correction; station elevation = NLDAS cell mean elevation):""")
for s in stations:
    print(f"  - NLDAS_{s['n']} ({s['label']} cell): easting {s['x']}, northing {s['y']}, elevation {s['elev']} m, lat {s['lat']:.4f}, lon {s['lon']:.4f}")
print(f"- Files: {MET_SDF} and the met_NLDAS_*.mdf files beside it. NLDAS rain files written outside the repo ({PRECIP_SDF}) and not used.")

### What to change in `Make_Sycamore_Model.ipynb`

In its Met section, point the weather station option at the file made here, and leave rain on the radar grids:

```python
met.hydrometstations['value'] = "../init_data/met/Master_Met.sdf"
```

Do **not** point `gaugestations` at the NLDAS rain file. Use `UTCOFFSET -7`, as for the radar.

### Still open (for Josh)
1. Does tRIBS correct station temperature and pressure for each node's elevation, and which option turns that on? What elevation should each station carry?
2. How many stations? Table 2 shows how much ground each one serves and how much cooler that ground is than the station value.
3. Does an NLDAS stamp mean the start or the end of the hour (section 8), and the same question for the radar files?
4. Spin-up: how many days or months before 10/01/2016 does the run need? That would mean more radar hours and a new `LOCAL_START` here.
5. With `RAINSOURCE = 1`, is the gauge rain file ignored?